# C11-neural-training — Practice p16 — Solution


**Type:** integrative (parts consume earlier results) · **Difficulty:** core · **Concepts:** autograd-training, torch-optimizers


The loss history is recorded before each step. Initial snapshots provide an
independent movement certificate, while gradients are inspected after the last
backward and before its optimizer step.


In [ ]:
import torch
import torch.nn as nn

def train_torch_xor(seed=20260804,epochs=500):
    torch.manual_seed(seed); torch.use_deterministic_algorithms(True); torch.set_default_dtype(torch.float64)
    X=torch.tensor([[-1.,-1.],[-1.,1.],[1.,-1.],[1.,1.]],dtype=torch.float64).repeat((24,1))
    y=torch.tensor([0,1,1,0],dtype=torch.long).repeat(24)
    model=nn.Sequential(nn.Linear(2,8),nn.ReLU(),nn.Linear(8,2)).to(dtype=torch.float64,device="cpu")
    initial=[p.detach().clone() for p in model.parameters()]; criterion=nn.CrossEntropyLoss()
    optimizer=torch.optim.Adam(model.parameters(),lr=.04); losses=torch.empty(epochs,dtype=torch.float64)
    all_grads=False
    for epoch in range(epochs):
        optimizer.zero_grad(set_to_none=True)  # PLAN017_MUTATION_TARGET: C11-p16-zero-grad
        logits=model(X); loss=criterion(logits,y); losses[epoch]=loss.detach(); loss.backward()
        all_grads=all(p.grad is not None and p.grad.shape==p.shape and torch.isfinite(p.grad).all() for p in optimizer.param_groups[0]["params"])
        optimizer.step()
    centers=X[:4]; predictions=model(centers).argmax(1).detach().cpu()
    movement=max(float(torch.linalg.vector_norm(p.detach()-q)) for p,q in zip(model.parameters(),initial))
    model._training_optimizer=optimizer
    return {"model":model,"losses":losses.detach().cpu(),"predictions":predictions,
            "max_parameter_movement":movement,"all_grads_present_last_backward":bool(all_grads)}

result_p16=train_torch_xor(); repeat_p16=train_torch_xor()


### Answer check


In [ ]:
# PLAN017_ANSWER_CHECK: C11-p16-training
assert set(result_p16)=={"model","losses","predictions","max_parameter_movement","all_grads_present_last_backward"}
assert result_p16["losses"].dtype==torch.float64 and result_p16["losses"].shape==(500,) and torch.isfinite(result_p16["losses"]).all()
model_check_p16=result_p16["model"]; optimizer_check_p16=model_check_p16._training_optimizer
assert result_p16["all_grads_present_last_backward"]
assert all(p.grad is not None and p.grad.shape==p.shape and torch.isfinite(p.grad).all() for p in model_check_p16.parameters())
assert len(optimizer_check_p16.state)==len(list(model_check_p16.parameters()))
assert all(int(optimizer_check_p16.state[p]["step"].item())==500 for p in model_check_p16.parameters())
torch.manual_seed(20260804)
initial_model_p16=nn.Sequential(nn.Linear(2,8),nn.ReLU(),nn.Linear(8,2)).to(dtype=torch.float64,device="cpu")
movement_check_p16=max(float(torch.linalg.vector_norm(p.detach()-q.detach())) for p,q in zip(model_check_p16.parameters(),initial_model_p16.parameters()))
assert abs(movement_check_p16-result_p16["max_parameter_movement"]) <= 1e-12 + 1e-12*abs(movement_check_p16)
assert movement_check_p16>.1 and result_p16["losses"][-1] < .2*result_p16["losses"][0]
assert torch.equal(result_p16["predictions"],torch.tensor([0,1,1,0]))
assert torch.allclose(result_p16["losses"],repeat_p16["losses"],atol=1e-9,rtol=1e-7)
for p,q in zip(model_check_p16.parameters(),repeat_p16["model"].parameters()): assert torch.allclose(p,q,atol=1e-9,rtol=1e-7)
